In [4]:
# ## 1. Imports
# Import required libraries for data handling and modeling

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Extra
import joblib
import gradio as gr

### 2. Load Dataset
##### Load the property listing data from a CSV file

In [5]:
df = pd.read_csv("horizon_properties_all_pages2.csv")


### 3. Clean the Price Column
##### Remove currency symbols and commas, convert to numeric values

In [6]:
df['Price_clean'] = df['Price'].str.replace(r'[^\d.]', '', regex=True)
df['Price_clean'] = pd.to_numeric(df['Price_clean'], errors='coerce')
df = df.dropna(subset=['Price_clean'])
df['Price'] = df['Price_clean']
df.drop(columns=['Price_clean'], inplace=True)

### 4. Convert Property Size to Square Meters
##### Handle acres and dimension-based formats like "50x30"

In [7]:
def convert_size(size):
    if pd.isna(size):
        return np.nan
    size = size.lower().strip()
    
    # Convert acres to square meters
    acre_match = re.search(r'([\d.]+)\s*acre', size)
    if acre_match:
        return float(acre_match.group(1)) * 4046.86

    # Convert dimensions (e.g., 50x30)
    dim_match = re.search(r'(\d+)[xX×](\d+)', size)
    if dim_match:
        return float(dim_match.group(1)) * float(dim_match.group(2))

    return np.nan

df['Size_m2'] = df['Property Size'].apply(convert_size)

### 5. Fill Missing Bedrooms
#### Replace missing values in the 'Bedrooms' column with the median


In [8]:
df['Bedrooms'] = df['Bedrooms'].fillna(df['Bedrooms'].median())


### 6. Filter Outliers and Irrelevant Entries
#### Keep records with realistic values only

In [9]:
df_basic = df[
    (df['Price'] > 10000) & (df['Price'] < 5000000) &
    (df['Size_m2'] > 100) & (df['Size_m2'] < 5000) &
    (df['Bedrooms'] > 0) & (df['Bedrooms'] < 10)
]

### 7. Encode Categorical Features
#### Use one-hot encoding for 'Property Type' and 'Property Status'

In [10]:
df_model_basic = pd.get_dummies(df_basic, columns=['Property Type', 'Property Status'], drop_first=True)


### 8. Define Features and Target
#### Select numeric and encoded features, and set 'Price' as the target variable

In [11]:
features = ['Bedrooms', 'Size_m2'] + \
           [col for col in df_model_basic.columns if 'Property Type_' in col or 'Property Status_' in col]

X = df_model_basic[features]
y = df_model_basic['Price']

### 9. Train Linear Regression Model
#### Split data into training and testing sets, fit model, and predict

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_basic = LinearRegression()
model_basic.fit(X_train, y_train)
y_pred_basic = model_basic.predict(X_test)

### 10. Evaluate Model Performance
#### Use RMSE and MAPE to assess model accuracy

In [20]:
rmse_basic = np.sqrt(mean_squared_error(y_test, y_pred_basic))
mape_basic = np.mean(np.abs((y_test - y_pred_basic) / y_test)) * 100

#### Save the model and features

In [21]:
# ## 11. Save Model and Columns for GUI
joblib.dump(model_basic, "house_price_model.pkl")
joblib.dump(features, "model_features.pkl")

['model_features.pkl']

#### 11. Display Results

In [22]:
print("Sample of Cleaned & Encoded Data:")
print(df_model_basic.head())
print("\nModel Coefficients:")
print(model_basic.coef_)
print("\nIntercept:", model_basic.intercept_)
print("\nRoot Mean Squared Error (RMSE):", rmse_basic)
print("Mean Absolute Percentage Error (MAPE):", mape_basic, "%")

Sample of Cleaned & Encoded Data:
                      Title     Price Property Size  Bedrooms  Size_m2  \
1   MEAN WOOD NDEKE PHASE 3  900000.0    28x28 sqft       3.0    784.0   
2                 SHANTUMBU  120000.0    20x30 sqft       3.0    600.0   
8    MEANWOOD CHAMBA VALLEY  450000.0    18x28 sqft       3.0    504.0   
18       LILAYI (YORK FARM)  360000.0    20x30 sqft       3.0    600.0   
19                 10 MILES  420000.0    40X30 sqft       4.0   1200.0   

    Property Type_Multi Family Home  Property Type_Plot  
1                             False               False  
2                             False                True  
8                             False                True  
18                            False                True  
19                            False               False  

Model Coefficients:
[ 1.03192606e+05 -1.34439996e+02  2.15905423e+05 -8.00512106e+05]

Intercept: 875346.7545826224

Root Mean Squared Error (RMSE): 1384869.3566086749
Mean

In [23]:
def predict_price(bedrooms, size, property_type, property_status):
    model = joblib.load("house_price_model.pkl")
    features = joblib.load("model_features.pkl")

    # Create base input with zeros
    input_data = {feat: 0 for feat in features}
    input_data['Bedrooms'] = bedrooms
    input_data['Size_m2'] = size

    # Handle one-hot encodings
    type_key = f"Property Type_{property_type}"
    status_key = f"Property Status_{property_status}"
    if type_key in input_data:
        input_data[type_key] = 1
    if status_key in input_data:
        input_data[status_key] = 1

    input_df = pd.DataFrame([input_data])
    pred = model.predict(input_df)[0]
    return f"Estimated Price: ZK {pred:,.0f}"

In [24]:
# Launch GUI
property_types = sorted([col.replace("Property Type_", "") for col in features if col.startswith("Property Type_")])
property_statuses = sorted([col.replace("Property Status_", "") for col in features if col.startswith("Property Status_")])

demo = gr.Interface(
    fn=predict_price,
    inputs=[
        gr.Number(label="Bedrooms"),
        gr.Number(label="Size (in m²)"),
        gr.Dropdown(property_types, label="Property Type"),
        gr.Dropdown(property_statuses, label="Property Status")
    ],
    outputs="text",
    title="House Price Estimator"
)

demo.launch()

Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.
